# CBIS-DDSM: indirme → preprocess → GPU eğitim (Harmonia)

Harici ortamda (Colab, başka PC, farklı klasör) çalışması için **yollar ortam değişkenleri ve otomatik keşifle** ayarlanır; sabit `C:\...` yok.

**Veri:** [CBIS-DDSM (TCIA)](https://www.cancerimagingarchive.net/collection/cbis-ddsm/) — genelde NBIA / web ile indirilir; aşağıda zip veya klasör yolu env veya keşifle verilir.

## 1) Depo kökünü ve yolları otomatik bağlama

Öncelik sırası:

1. `HARMONIA_REPO_ROOT` ortam değişkeni
2. Çalışma dizininden ve üst klasörlerden `harmonia_vision/` paketini arama
3. (İsteğe bağlı) `pip install -e .` ile kurulu paketin konumu — yoksa yok sayılır

Çıktı ve veri için: `HARMONIA_DATASET_ROOT`, `HARMONIA_OUT_ROOT`, `HARMONIA_CHECKPOINT` tanımlıysa onlar kullanılır; değilse depo köküne göre varsayılan alt yollar.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import sys
from pathlib import Path


def find_harmonia_repo_root() -> Path:
    """harmonia_vision paketini içeren depo kökünü bulur."""
    env = os.environ.get("HARMONIA_REPO_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "harmonia_vision").is_dir():
            return p
        raise FileNotFoundError(f"HARMONIA_REPO_ROOT geçersiz (harmonia_vision yok): {p}")

    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "harmonia_vision" / "__init__.py").is_file():
            return p.resolve()

    spec = importlib.util.find_spec("harmonia_vision")
    if spec and spec.origin:
        pkg = Path(spec.origin).resolve().parent
        if pkg.name == "harmonia_vision":
            return pkg.parent

    raise FileNotFoundError(
        "harmonia_vision bulunamadı. Çözüm: os.environ['HARMONIA_REPO_ROOT']=... "
        "veya Jupyter'de %cd ile depo köküne gidin (cwd'den üst klasörler taranır)."
    )


REPO_ROOT = find_harmonia_repo_root()
os.environ["PYTHONPATH"] = str(REPO_ROOT)
rp = str(REPO_ROOT)
if rp not in sys.path:
    sys.path.insert(0, rp)


def pick_path(env_name: str, default: Path) -> Path:
    v = os.environ.get(env_name, "").strip()
    return Path(v).expanduser().resolve() if v else default


DATASET_ROOT = pick_path("HARMONIA_DATASET_ROOT", REPO_ROOT / "data" / "CBIS-DDSM")
OUT_ROOT = pick_path("HARMONIA_OUT_ROOT", REPO_ROOT / "harmonia_processed_notebook")
CKPT = pick_path("HARMONIA_CHECKPOINT", REPO_ROOT / "harmonia_checkpoints" / "notebook_centralized_long.pth")

print("REPO_ROOT", REPO_ROOT)
print("DATASET_ROOT", DATASET_ROOT)
print("OUT_ROOT", OUT_ROOT)
print("CKPT", CKPT)

import torch

print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print("  " + torch.cuda.get_device_name(0))
else:
    print("  (CPU — eğitim yavaşlar)")

## 2) Veri seti (zip veya hazır klasör)

İndirdiğiniz arşivi açmak için `HARMONIA_EXTRACT_ZIP` (tek zip dosyası yolu) verebilirsiniz; yoksa sadece `DATASET_ROOT` altında `csv/` olduğunu doğrularız.

In [ ]:
import zipfile

zip_env = os.environ.get("HARMONIA_EXTRACT_ZIP", "").strip()
if zip_env:
    zpath = Path(zip_env).expanduser().resolve()
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(DATASET_ROOT)
    print("Çıkarıldı:", DATASET_ROOT)

csv_ok = (DATASET_ROOT / "csv" / "dicom_info.csv").is_file()
print("csv/dicom_info.csv:", "OK" if csv_ok else "EKSİK — DATASET_ROOT veya zip yolunu kontrol edin")
if not csv_ok:
    print("İpucu: HARMONIA_DATASET_ROOT ile TCIA çıkartma kökünü gösterin.")

## 3) Preprocess

İlk denemede `HARMONIA_MAX_GROUPS=48` gibi küçük tutun (sadece bu not defteri için `os.environ` ile aşağıda okunuyor).

In [ ]:
import subprocess

OUT_ROOT.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable,
    "-m",
    "harmonia_vision.data_pipeline",
    "--dataset-root",
    str(DATASET_ROOT),
    "--out-root",
    str(OUT_ROOT),
    "--split-mode",
    "iid",
]
mg = os.environ.get("HARMONIA_MAX_GROUPS", "").strip()
if mg:
    cmd.extend(["--max-groups", mg])

print(" ".join(cmd))
r = subprocess.run(
    cmd,
    cwd=str(REPO_ROOT),
    env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONPATH": str(REPO_ROOT)},
)
if r.returncode != 0:
    raise RuntimeError(f"preprocess çıkış kodu: {r.returncode}")

## 4) GPU ile merkezi eğitim (`benchmark`)

Uzun koşu: `HARMONIA_PRESET=long` (varsayılan aşağıda). Kısa için `quick`.

In [ ]:
manifest = OUT_ROOT / "manifest_iid.csv"
if not manifest.is_file():
    raise FileNotFoundError(f"Manifest yok (preprocess bitmeden eğitim başlamaz): {manifest}")

CKPT.parent.mkdir(parents=True, exist_ok=True)
preset = os.environ.get("HARMONIA_PRESET", "long").strip() or "long"

train_cmd = [
    sys.executable,
    "-m",
    "harmonia_vision.benchmark",
    "--mode",
    "centralized",
    "--data-root",
    str(OUT_ROOT),
    "--preset",
    preset,
    "--checkpoint",
    str(CKPT),
]
if os.environ.get("HARMONIA_NO_VAL_SPLIT", "").strip().lower() in ("1", "true", "yes"):
    train_cmd.append("--no-val-split")

print(" ".join(train_cmd))
r = subprocess.run(
    train_cmd,
    cwd=str(REPO_ROOT),
    env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONPATH": str(REPO_ROOT)},
)
print("exit:", r.returncode)

## Ortam değişkenleri özeti

| Değişken | Anlamı |
|----------|--------|
| `HARMONIA_REPO_ROOT` | Depo kökü (zorunlu değil; otomatik bulunamazsa gerekir) |
| `HARMONIA_DATASET_ROOT` | CBIS-DDSM çıkartma kökü (`csv/` içinde) |
| `HARMONIA_OUT_ROOT` | Preprocess `.npy` çıktısı |
| `HARMONIA_CHECKPOINT` | Kaydedilecek `.pth` |
| `HARMONIA_EXTRACT_ZIP` | İsteğe bağlı: zip’ten `DATASET_ROOT`’a aç |
| `HARMONIA_MAX_GROUPS` | Örn. `48` — kısa smoke preprocess |
| `HARMONIA_PRESET` | `quick` / `standard` / `long` |
| `HARMONIA_NO_VAL_SPLIT` | `1` ise val ayrımı yok (debug) |

Colab örneği: önce depoyu klonlayın, sonra `os.environ["HARMONIA_REPO_ROOT"] = "/content/model_0"` ve veri yolunu ayarlayın.